In [1]:
import os
import string
import random
import siphash
import math
import itertools


In [2]:
import json
from pathlib import Path

with Path("confusables.json").open(encoding="utf-8") as file:
    data = json.load(file)

character = "a"
target_code_points = [f"{ord(char):04X}" for char in character]

# UTS #39 mappings are directional: find source characters that map to 'a'.
confusables = [
    mapping["source"]
    for mapping in data["mappings"]
    if mapping["target"]["codePoints"] == target_code_points
]

print(f"Confusables mapping to {character!r} (U+{target_code_points[0]}):")
for item in confusables:
    code_points = " ".join(f"U+{point}" for point in item["codePoints"])
    print(f"{ascii(item['text'])} ({code_points})")

Confusables mapping to 'a' (U+0061):
'\u237a' (U+237A)
'\uff41' (U+FF41)
'\U0001d41a' (U+1D41A)
'\U0001d44e' (U+1D44E)
'\U0001d482' (U+1D482)
'\U0001d4b6' (U+1D4B6)
'\U0001d4ea' (U+1D4EA)
'\U0001d51e' (U+1D51E)
'\U0001d552' (U+1D552)
'\U0001d586' (U+1D586)
'\U0001d5ba' (U+1D5BA)
'\U0001d5ee' (U+1D5EE)
'\U0001d622' (U+1D622)
'\U0001d656' (U+1D656)
'\U0001d68a' (U+1D68A)
'\U0001df5a' (U+1DF5A)
'\u0251' (U+0251)
'\uab64' (U+AB64)
'\u03b1' (U+03B1)
'\U0001d6c2' (U+1D6C2)
'\U0001d6fc' (U+1D6FC)
'\U0001d736' (U+1D736)
'\U0001d770' (U+1D770)
'\U0001d7aa' (U+1D7AA)
'\u0430' (U+0430)


In [3]:
import json
import random
from pathlib import Path

sentence = "This is a test phrase. There are many a's in this sentence."

with Path("confusables.json").open(encoding="utf-8") as file:
    data = json.load(file)

homoglyphs = [
    mapping["source"]["text"]
    for mapping in data["mappings"]
    if mapping["target"]["codePoints"] == ["0061"]
]

transformed_sentence = "".join(
    random.choice(homoglyphs) if character == "a" else character
    for character in sentence
)
print(transformed_sentence)

This is ⍺ test phr𝞪se. There ⍺re m𝒂ny 𝚊's in this sentence.


In [ ]:
import hmac
import hashlib
import siphash


def siphash_text(text: str, password: str) -> str:
    # SipHash requires a 16-byte key; derive one from the password.
    key = hashlib.blake2b(password.encode("utf-8"), digest_size=16).digest()
    message = text.encode("utf-8")
    return f"{siphash.SipHash_2_4(key, message).hash():016x}"


def verify_siphash(text: str, password: str, expected_hash: str) -> bool:
    # SipHash cannot be reversed; check a candidate text and password instead.
    try:
        expected = bytes.fromhex(expected_hash)
    except ValueError:
        return False
    if len(expected) != 8:
        return False
    actual = bytes.fromhex(siphash_text(text, password))
    return hmac.compare_digest(actual, expected)


text = "This is a test text. it will be much longer in practice. The password will be used to generate a SipHash of this text."
password = "password123"
print("SipHash:", siphash_text(text, password))

print("Comparing with correct password:", verify_siphash(text, password, siphash_text(text, password)))

